In [35]:
#STEP 1
#Import libraries

import os
from dotenv import load_dotenv
import json
import sqlite3
from openai import OpenAI
from datetime import datetime
import gradio as gr

In [36]:
#Set up OpenAI API key
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [37]:
#Step 2: Create SQLite database and tables
DB = "finance_tracker.db"
with sqlite3.connect(DB) as conn:
  cursor = conn.cursor()
  
  #Table 1: salary
  cursor.execute('''
    CREATE TABLE IF NOT EXISTS salary (
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      amount REAL NOT NULL,
      month TEXT ,
      year INTEGER ,
      created_at TEXT
    )
  ''')
  
  #Table 2: expenses
  cursor.execute('''
    CREATE TABLE IF NOT EXISTS expenses (
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      amount REAL NOT NULL,
      category TEXT ,
      description TEXT,
      date TEXT,
      created_at TEXT
    )
  ''')
  
  conn.commit()
  
  
  print("Both tables created successfully!")

Both tables created successfully!


In [38]:
#  Step 2 : Implement the Tool Functions

#Function 1 : set_salary
def set_salary(amount, month, year):
    conn = sqlite3.connect(DB)
    cursor = conn.cursor()
    
    cursor.execute("INSERT INTO salary (amount, month, year, created_at) VALUES (?, ?, ?, ?)",
                     (amount, month, year, datetime.now().isoformat()))
    conn.commit()
    conn.close()
    return f"Salary of ${amount:,.2f} set for {month} {year}."

#Function 2 : log_expense
def log_expense(amount, category, description, date=None):
    if not date:
        date = datetime.now().strftime("%Y-%m-%d")
        
    category = category.lower()
    conn = sqlite3.connect(DB)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO expenses (amount, category, description, date, created_at) VALUES (?, ?, ?, ?, ?)",
                     (amount, category, description, date, datetime.now().isoformat()))
    
    conn.commit()
    
    #update balance after logging expense
    cursor.execute("Select SUM(amount) from expenses")
    result = cursor.fetchone()
    spent = result[0] if result and result[0] else 0
    
    cursor.execute("select amount from salary order by id DESC limit 1")
    sal_result = cursor.fetchone()
    salary = sal_result[0] if sal_result else 0
    conn.close()
    return f"Expense of ${amount:,.2f} logged for {category}."
    
    
    # Function 3 : get_balance
def get_balance():
    conn = sqlite3.connect(DB)
    cursor = conn.cursor()

    cursor.execute("Select SUM(amount) from expenses")
    result = cursor.fetchone()
    spent = result[0] if result and result[0] else 0
    
    cursor.execute("select amount from salary order by id DESC limit 1")
    sal_result = cursor.fetchone()
    salary = sal_result[0] if sal_result else 0
    conn.close()

    return f"Salary: ${salary:,.2f}, Total spent: {spent:,.2f}, Remaining balance: ${salary - spent:,.2f}"
    
    
# Function 4 : get_expense_summary
def get_expense_summary():
    conn = sqlite3.connect(DB)
    cursor = conn.cursor()

    cursor.execute("select amount from salary order by id DESC limit 1")
    sal_result = cursor.fetchone()
    salary = sal_result[0] if sal_result else 0

    cursor.execute("SELECT category, SUM(amount) from expenses group by category")
    rows = cursor.fetchall()

    conn.close()

    if not rows: return "no expenses found"

    output = "Expense Summary : \n"
    for category, amount in rows:
        percentage = (amount / salary * 100) if salary > 0 else 0
        output += f"- {category}: ${amount:,.2f} ({percentage:.1f}% of salary)\n"

    return output


                    
            
        
    

In [39]:
# Step 3 - Describe the tools to the AI

tools = [
    {
        "type": "function",
        "function": {
            "name": "set_salary",
            "description": "Set user's monthly salary",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number"},
                    "month": {"type": "string"},
                    "year": {"type": "number"}
                },
                "required": ["amount", "month", "year"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "log_expense",
            "description": "Log an expense with category, description and optional date",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number"},
                    "category": {"type": "string"},
                    "description": {"type": "string"},
                    "date": {"type": "string", "description": "Optional date in YYYY-MM-DD format"}
                },
                "required": ["amount", "category", "description"]
            }
        }
    },
    {
        
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get current salary, total spent and remaining balance",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_expense_summary",
            "description": "Get spending breakdown by category",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
    
]

In [40]:
# System prompt

system_message = """
You are a smart personal finance assistant that helps users manage their salary and expenses.

Your main task is to understand the user's message and decide which finance tool should be used.

When the user provides salary details:
- identify the salary amount
- identify the month
- identify the year

When the user provides expense details:
- identify the expense amount
- identify the category
- identify the description
- identify the date, if mentioned

Use suitable expense categories such as:
food, rent, groceries, entertainment, transport, health, subscription

Category examples:
- "spent $12 on coffee" should be saved under food
- "paid $500 for rent" should be saved under rent
- "Netflix subscription $20" should be saved under subscription
- "bus ticket $3" should be saved under transport
- "bought medicine for $15" should be saved under health

Important rule:
Whenever the user gives salary or expense information, always use the correct tool to save or process it.
Do not give a normal text reply without calling a tool when the information needs to be recorded.
"""

In [41]:
# Step 5 : The tool calling loop - 
# This is where we process the AI's response and execute any tool calls it makes.

def handle_tool_calls(reply):
    tool_messages = []
    
    for tool_call in reply.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        
        if name == "set_salary":
            result = set_salary(**args)
        elif name == "log_expense":
            result = log_expense(**args)
        elif name == "get_balance":
            result = get_balance()
        elif name == "get_expense_summary":
            result = get_expense_summary()
        else:
            result = f"Unknown tool: {name}"
            
        tool_messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    
    return tool_messages    

In [42]:
# Main chat function - This is the main function that will be 
# called when the user sends a message. 
# It processes the message, handles tool calls, and returns the final response to the user.
def chat(message, history):
    try:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": message}
        ]
        
        # Initial API call to get AI response
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        reply = response.choices[0].message
        
        while reply.tool_calls:
            messages.append({
                "role": "assistant", 
                "content": reply.content,
                "tool_calls": reply.tool_calls
            })
            
            messages.extend(handle_tool_calls(reply))
            
            response = openai.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
                tool_choice="auto"
            )
            
            reply = response.choices[0].message
            
        return reply.content
    except Exception as e:
        return f"Error: {str(e)}"
        

In [48]:
conn =sqlite3.connect(DB)
cursor = conn.cursor()

cursor.execute("DELETE FROM salary")
cursor.execute("DELETE FROM expenses")

conn.commit()
conn.close()

print("Database cleared")

Database cleared


In [47]:
# Step 6: Create Gradio interface - This is where we set up the user interface for our personal
# - finance tracker using Gradio.
gr.ChatInterface(
    fn=chat, 
    title="Personal Finance Tracker",
    description="A smart assistant to help you track your income and expenses.",
    examples=[
        "Set my salary to 4000 for June 2026",
        "I spent 50 on food for lunch",
        "How much money do I have left?",
        "Show my expense summary" 
    ]
    ).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
